# Cleaning and EDA over Datasets

# Dataset 1 — Zomato (7105 rows → 7105 rows)  
**What we found:** Two junk index columns Unnamed: 0 and Unnamed: 0.1 were duplicates of the dataframe index carrying no information. The rate (out of 5) column had 68 missing values — all genuine missing ratings, no corrupt strings. The avg cost (two people) column had 57 missing values.  
**What we did and why:** Dropped both Unnamed columns since they add no analytical value. Filled 68 rating nulls with the median value 3.5 — median is safer than mean here because the distribution is slightly skewed. Filled 57 cost nulls with median 400.0 — the cost column has a wide range (40 to 6000) with outliers pulling the mean to 540, so median is more representative. Converted online_order and table_booking from Yes/No strings to 1/0 integers for MySQL compatibility. Added a source column with value 'zomato' to track origin after merging with Swiggy. Added city = 'Bangalore' since the entire Zomato dataset is from Bangalore.

  


# Dataset 2 — Swiggy (8680 rows → 8680 rows)
**What we found:** Perfectly clean — zero nulls across all 10 columns. All data types were correct. Delivery time had a healthy range of 20 to 109 minutes with mean 54, representing restaurant-reported estimated delivery times.  
**What we did and why:** Dropped the ID column and replaced with a warehouse surrogate key. Renamed all columns to match the warehouse schema. Added source = 'swiggy' for origin tracking. Added restaurant_type, table_booking, online_order as None since Swiggy doesn't provide these — they'll be NULL in MySQL which is acceptable.

# Zomato + Swiggy → Dim_Restaurant (15785 rows)
**What we did and why:** Instead of joining (impossible — no shared key), we performed a UNION using pd.concat. This merges both platforms into one unified restaurant dimension table. Each row knows its origin via the source column. A surrogate restaurant_id (1 to 15785) was generated as the primary key since neither dataset had a reliable unique restaurant identifier.

# Dataset 3 — Food Delivery Time (1000 rows → 1000 rows)
**What we found:** 30 rows had Weather as NaN. On closer inspection these were partial nulls — Delivery_Time_min was still valid on all 30 rows (values like 32, 85, 109) making them valuable fact records worth keeping. Only 1 row also had Time_of_Day null and only 1 row had Courier_Experience_yrs null.  
**What we did and why:** Filled Weather nulls with 'Clear' — the mode with 470 out of 1000 occurrences (47%), so Clear is the statistically dominant weather condition. Filled Courier_Experience_yrs null with median 5.0. Filled remaining Traffic_Level and Time_of_Day nulls with their respective modes. Chose filling over dropping because Delivery_Time_min was intact on all 30 rows — dropping would have wasted valid delivery records.  
Added delay_flag as a derived column — 1 if Delivery_Time_min > 45 else 0. This is the single most important derived column in the entire warehouse, directly powering the late delivery KPI on the dashboard.  
Key insight from EDA: 664 out of 1000 orders (66.4%) exceeded 45 minutes. This is a significant operational finding that becomes the central story of the delivery performance dashboard.
Added rider_id as a surrogate key grouped by Vehicle_Type and Courier_Experience_yrs combination.

# Dataset 4 — Online Food Orders / Customers (388 rows → 388 rows)
**What we found:** Perfectly clean — zero nulls, all columns present and correctly typed. Monthly Income is a string category (e.g. "Below Rs.10000", "25001 to 50000") which is intentional. 301 out of 388 customers (77.6%) order online. 317 out of 388 (81.7%) gave positive feedback.   
**What we did and why:** Encoded Output (Yes/No → 1/0) and Feedback (Positive/Negative → 1/0) for MySQL integer storage and easier aggregation in Superset. Generated surrogate customer_id as primary key. Renamed columns to snake_case warehouse schema naming convention.

# Derived tables built during cleaning
**Dim_Rider** — extracted from the delivery dataset by grouping on Vehicle_Type and Courier_Experience_yrs, computing average delivery time and total deliveries per rider profile. Resulted in a compact rider dimension.  
**Dim_Time** — extracted unique Time_of_Day values (Morning, Afternoon, Evening, Night) and mapped each to a representative hour (9, 13, 18, 22). is_weekend defaulted to 0 since the source dataset had no date information.  
**Dim_Location** — built from the customers dataset using latitude, longitude, and pin_code. All locations are in Bangalore area based on coordinate range.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
zomato = pd.read_csv('zomato.csv')
swiggy = pd.read_csv('swiggy.csv')
delivery = pd.read_csv('Food_Delivery_Times.csv')
customers = pd.read_csv('onlinefoods.csv')

print("Zomato:", zomato.shape)
print("Swiggy:", swiggy.shape)
print("Delivery:", delivery.shape)
print("Customers:", customers.shape)

Zomato: (7105, 12)
Swiggy: (8680, 10)
Delivery: (1000, 9)
Customers: (388, 12)


In [30]:
print("===== ZOMATO =====")
zomato.head(3)

===== ZOMATO =====


,name,restaurant_type,avg_rating,num_ratings,avg_cost_two,online_order,table_booking,cuisine_type,area,address,source,city,typical_delivery_time
0,#FeelTheROLL,Quick Bites,3.4,7,200.0,0,0,Fast Food,Bellandur,Bellandur,zomato,Bangalore,None
1,#L-81 Cafe,Quick Bites,3.9,48,400.0,1,0,"Fast Food, Beverages","Byresandra,Tavarekere,Madiwala",HSR,zomato,Bangalore,None
2,#refuel,Cafe,3.7,37,400.0,1,0,"Cafe, Beverages",Bannerghatta Road,Bannerghatta Road,zomato,Bangalore,None


In [4]:
print("\n===== SWIGGY =====")
swiggy.head(3)


===== SWIGGY =====


,ID,Area,City,Restaurant,Price,Avg ratings,Total ratings,Food type,Address,Delivery time
0,211,Koramangala,Bangalore,Tandoor Hut,300.0,4.4,100,"Biryani,Chinese,North Indian,South Indian",5Th Block,59
1,221,Koramangala,Bangalore,Tunday Kababi,300.0,4.1,100,"Mughlai,Lucknowi",5Th Block,56
2,246,Jogupalya,Bangalore,Kim Lee,650.0,4.4,100,Chinese,Double Road,50


In [5]:
print("\n===== DELIVERY =====")
delivery.head(3)


===== DELIVERY =====


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59


In [6]:
print("\n===== CUSTOMERS =====")
customers.head(3)


===== CUSTOMERS =====


,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,latitude,longitude,Pin code,Output,Feedback
0,20,Female,Single,Student,No Income,Post Graduate,4,12.9766,77.5993,560001,Yes,Positive
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,12.9770,77.5773,560009,Yes,Positive
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,12.9551,77.6593,560017,Yes,Negative


In [7]:
for name, df in [("Zomato", zomato), ("Swiggy", swiggy), ("Delivery", delivery), ("Customers", customers)]:
    print(f"\n===== {name} NULLS =====")
    print(df.isnull().sum())


===== Zomato NULLS =====
Unnamed: 0.1              0
Unnamed: 0                0
restaurant name           0
restaurant type           0
rate (out of 5)          68
num of ratings            0
avg cost (two people)    57
online_order              0
table booking             0
cuisines type             0
area                      0
local address             0
dtype: int64

===== Swiggy NULLS =====
ID               0
Area             0
City             0
Restaurant       0
Price            0
Avg ratings      0
Total ratings    0
Food type        0
Address          0
Delivery time    0
dtype: int64

===== Delivery NULLS =====
Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Experience_yrs    30
Delivery_Time_min          0
dtype: int64

===== Customers NULLS =====
Age                           0
Gender                        0
M

In [8]:
for name, df in [("Zomato", zomato), ("Swiggy", swiggy), ("Delivery", delivery), ("Customers", customers)]:
    print(f"\n===== {name} DTYPES =====")
    print(df.dtypes)


===== Zomato DTYPES =====
Unnamed: 0.1               int64
Unnamed: 0                 int64
restaurant name           object
restaurant type           object
rate (out of 5)          float64
num of ratings             int64
avg cost (two people)    float64
online_order              object
table booking             object
cuisines type             object
area                      object
local address             object
dtype: object

===== Swiggy DTYPES =====
ID                 int64
Area              object
City              object
Restaurant        object
Price            float64
Avg ratings      float64
Total ratings      int64
Food type         object
Address           object
Delivery time      int64
dtype: object

===== Delivery DTYPES =====
Order_ID                    int64
Distance_km               float64
Weather                    object
Traffic_Level              object
Time_of_Day                object
Vehicle_Type               object
Preparation_Time_min        int64
Couri

In [9]:
print(zomato['rate (out of 5)'].unique()[:30])
print("\nUnnamed cols sample:")
print(zomato[['Unnamed: 0.1', 'Unnamed: 0']].head())

[3.4 3.9 3.7 2.7 2.8 4.1 3.2 3.5 4.4 4.2 4.  4.3 3.1 3.6 3.3 2.9 3.  3.8
 2.5 nan 4.6 1.8 2.4 4.5 4.9 2.1 4.7 2.6 4.8 2. ]

Unnamed cols sample:
   Unnamed: 0.1  Unnamed: 0
0             0           0
1             1           1
2             2           2
3             3           3
4             4           4


In [10]:
print(delivery[delivery['Weather'].isnull()])

     Order_ID  Distance_km Weather Traffic_Level Time_of_Day Vehicle_Type  \
42        313         0.99     NaN        Medium     Evening         Bike   
71        494         4.17     NaN           Low     Evening      Scooter   
153       430         2.78     NaN          High   Afternoon      Scooter   
167       690         4.15     NaN        Medium     Morning          Car   
168       315        16.80     NaN          High     Morning          Car   
203       558         1.06     NaN           Low   Afternoon      Scooter   
225         3        14.77     NaN          High     Morning      Scooter   
230       345         8.27     NaN          High   Afternoon      Scooter   
257       917         6.24     NaN        Medium     Evening         Bike   
281       798        11.52     NaN        Medium     Morning      Scooter   
343       126         8.83     NaN        Medium     Evening      Scooter   
353        70        19.74     NaN           Low         NaN          Car   

In [11]:
print("Zomato rate unique values:")
print(zomato['rate (out of 5)'].value_counts(dropna=False).head(20))

Zomato rate unique values:
rate (out of 5)
2.8    749
3.7    618
3.6    573
3.8    533
3.4    519
3.5    507
3.9    504
3.3    502
3.2    429
4.0    378
3.1    317
4.1    292
3.0    209
4.2    209
4.3    166
2.9    164
4.4    107
NaN     68
2.7     65
4.5     47
Name: count, dtype: int64


In [12]:
null_rows = delivery[delivery['Weather'].isnull()]
print(f"Null rows count: {len(null_rows)}")
print(null_rows)

Null rows count: 30
     Order_ID  Distance_km Weather Traffic_Level Time_of_Day Vehicle_Type  \
42        313         0.99     NaN        Medium     Evening         Bike   
71        494         4.17     NaN           Low     Evening      Scooter   
153       430         2.78     NaN          High   Afternoon      Scooter   
167       690         4.15     NaN        Medium     Morning          Car   
168       315        16.80     NaN          High     Morning          Car   
203       558         1.06     NaN           Low   Afternoon      Scooter   
225         3        14.77     NaN          High     Morning      Scooter   
230       345         8.27     NaN          High   Afternoon      Scooter   
257       917         6.24     NaN        Medium     Evening         Bike   
281       798        11.52     NaN        Medium     Morning      Scooter   
343       126         8.83     NaN        Medium     Evening      Scooter   
353        70        19.74     NaN           Low        

In [13]:
print("Swiggy delivery time stats:")
print(swiggy['Delivery time'].describe())
print("\nDelivery dataset delivery time stats:")
print(delivery['Delivery_Time_min'].describe())

Swiggy delivery time stats:
count    8680.000000
mean       53.967051
std        14.292335
min        20.000000
25%        44.000000
50%        53.000000
75%        64.000000
max       109.000000
Name: Delivery time, dtype: float64

Delivery dataset delivery time stats:
count    1000.000000
mean       56.732000
std        22.070915
min         8.000000
25%        41.000000
50%        55.500000
75%        71.000000
max       153.000000
Name: Delivery_Time_min, dtype: float64


In [14]:
print("Zomato rate stats:")
print(zomato['rate (out of 5)'].describe())

Zomato rate stats:
count    7037.000000
mean        3.514253
std         0.463249
min         1.800000
25%         3.200000
50%         3.500000
75%         3.800000
max         4.900000
Name: rate (out of 5), dtype: float64


In [15]:
print("Weather mode:", delivery['Weather'].mode()[0])
print("Weather value counts:\n", delivery['Weather'].value_counts())
print("\nCourier_Experience median:", delivery['Courier_Experience_yrs'].median())

Weather mode: Clear
Weather value counts:
 Weather
Clear    470
Rainy    204
Foggy    103
Snowy     97
Windy     96
Name: count, dtype: int64

Courier_Experience median: 5.0


In [16]:
print("Avg cost nulls:", zomato['avg cost (two people)'].isnull().sum())
print("Avg cost median:", zomato['avg cost (two people)'].median())
print("Avg cost distribution:\n", zomato['avg cost (two people)'].describe())

Avg cost nulls: 57
Avg cost median: 400.0
Avg cost distribution:
 count    7048.000000
mean      540.286464
std       462.902305
min        40.000000
25%       300.000000
50%       400.000000
75%       600.000000
max      6000.000000
Name: avg cost (two people), dtype: float64


In [17]:
print("online_order values:\n", zomato['online_order'].value_counts())
print("\ntable_booking values:\n", zomato['table booking'].value_counts())

online_order values:
 online_order
Yes    3727
No     3378
Name: count, dtype: int64

table_booking values:
 table booking
No     6361
Yes     744
Name: count, dtype: int64


In [18]:
print("Monthly Income categories:\n", customers['Monthly Income'].value_counts())
print("\nOutput values:\n", customers['Output'].value_counts())
print("\nFeedback values:\n", customers['Feedback'].value_counts())

Monthly Income categories:
 Monthly Income
No Income          187
25001 to 50000      69
More than 50000     62
10001 to 25000      45
Below Rs.10000      25
Name: count, dtype: int64

Output values:
 Output
Yes    301
No      87
Name: count, dtype: int64

Feedback values:
 Feedback
Positive     317
Negative      71
Name: count, dtype: int64


cleaning zomato dataset

In [19]:
# drop junk index columns
zomato.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], inplace=True)

# fill nulls
zomato['rate (out of 5)'].fillna(3.5, inplace=True)
zomato['avg cost (two people)'].fillna(400.0, inplace=True)

# encode Yes/No to 1/0
zomato['online_order'] = zomato['online_order'].map({'Yes': 1, 'No': 0})
zomato['table booking'] = zomato['table booking'].map({'Yes': 1, 'No': 0})

# add source column
zomato['source'] = 'zomato'

# rename to warehouse schema
zomato.rename(columns={
    'restaurant name'      : 'name',
    'restaurant type'      : 'restaurant_type',
    'rate (out of 5)'      : 'avg_rating',
    'num of ratings'       : 'num_ratings',
    'avg cost (two people)': 'avg_cost_two',
    'online_order'         : 'online_order',
    'table booking'        : 'table_booking',
    'cuisines type'        : 'cuisine_type',
    'area'                 : 'area',
    'local address'        : 'address'
}, inplace=True)

print("Zomato cleaned:", zomato.shape)
zomato.head()

Zomato cleaned: (7105, 11)


/tmp/ipykernel_8171/1707000120.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  zomato['rate (out of 5)'].fillna(3.5, inplace=True)
/tmp/ipykernel_8171/1707000120.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', t

,name,restaurant_type,avg_rating,num_ratings,avg_cost_two,online_order,table_booking,cuisine_type,area,address,source
0,#FeelTheROLL,Quick Bites,3.4,7,200.0,0,0,Fast Food,Bellandur,Bellandur,zomato
1,#L-81 Cafe,Quick Bites,3.9,48,400.0,1,0,"Fast Food, Beverages","Byresandra,Tavarekere,Madiwala",HSR,zomato
2,#refuel,Cafe,3.7,37,400.0,1,0,"Cafe, Beverages",Bannerghatta Road,Bannerghatta Road,zomato
3,'@ Biryani Central,Casual Dining,2.7,135,550.0,1,0,"Biryani, Mughlai, Chinese",Marathahalli,Marathahalli,zomato
4,'@ The Bbq,Casual Dining,2.8,40,700.0,1,0,"BBQ, Continental, North Indian, Chinese, Bever...",Bellandur,Bellandur,zomato


cleaning swiggy dataset

In [20]:
# add source column
swiggy['source'] = 'swiggy'

# rename to warehouse schema
swiggy.rename(columns={
    'Restaurant'    : 'name',
    'Food type'     : 'cuisine_type',
    'Avg ratings'   : 'avg_rating',
    'Total ratings' : 'num_ratings',
    'Price'         : 'avg_cost_two',
    'Area'          : 'area',
    'Address'       : 'address',
    'City'          : 'city',
    'Delivery time' : 'typical_delivery_time'
}, inplace=True)

# drop ID column (we'll generate our own surrogate key)
swiggy.drop(columns=['ID'], inplace=True)

In [21]:
print("Swiggy cleaned:", swiggy.shape)
swiggy.head(2)

Swiggy cleaned: (8680, 10)


,area,city,name,avg_cost_two,avg_rating,num_ratings,cuisine_type,address,typical_delivery_time,source
0,Koramangala,Bangalore,Tandoor Hut,300.0,4.4,100,"Biryani,Chinese,North Indian,South Indian",5Th Block,59,swiggy
1,Koramangala,Bangalore,Tunday Kababi,300.0,4.1,100,"Mughlai,Lucknowi",5Th Block,56,swiggy


merging zomato and swiggy into Dim_Restaurant

In [22]:
# align columns — add missing cols as NaN before concat
zomato['city'] = 'Bangalore'  # zomato dataset is Bangalore
zomato['typical_delivery_time'] = None
swiggy['restaurant_type'] = None
swiggy['table_booking'] = None
swiggy['online_order'] = None

dim_restaurant = pd.concat([zomato, swiggy], ignore_index=True)

# add surrogate primary key
dim_restaurant.insert(0, 'restaurant_id', range(1, len(dim_restaurant) + 1))

print("Dim_Restaurant shape:", dim_restaurant.shape)
print("Columns:", dim_restaurant.columns.tolist())
dim_restaurant.head(3)

Dim_Restaurant shape: (15785, 14)
Columns: ['restaurant_id', 'name', 'restaurant_type', 'avg_rating', 'num_ratings', 'avg_cost_two', 'online_order', 'table_booking', 'cuisine_type', 'area', 'address', 'source', 'city', 'typical_delivery_time']


,restaurant_id,name,restaurant_type,avg_rating,num_ratings,avg_cost_two,online_order,table_booking,cuisine_type,area,address,source,city,typical_delivery_time
0,1,#FeelTheROLL,Quick Bites,3.4,7,200.0,0,0,Fast Food,Bellandur,Bellandur,zomato,Bangalore,None
1,2,#L-81 Cafe,Quick Bites,3.9,48,400.0,1,0,"Fast Food, Beverages","Byresandra,Tavarekere,Madiwala",HSR,zomato,Bangalore,None
2,3,#refuel,Cafe,3.7,37,400.0,1,0,"Cafe, Beverages",Bannerghatta Road,Bannerghatta Road,zomato,Bangalore,None


clean Delivery dataset

In [23]:
# fill nulls using assignment instead of inplace
delivery['Weather'] = delivery['Weather'].fillna('Clear')
delivery['Courier_Experience_yrs'] = delivery['Courier_Experience_yrs'].fillna(5.0)
delivery['Traffic_Level'] = delivery['Traffic_Level'].fillna(delivery['Traffic_Level'].mode()[0])
delivery['Time_of_Day'] = delivery['Time_of_Day'].fillna(delivery['Time_of_Day'].mode()[0])

# add delay_flag derived column
delivery['delay_flag'] = (delivery['Delivery_Time_min'] > 45).astype(int)

# add surrogate rider_id
delivery['rider_id'] = delivery.groupby(
    ['Vehicle_Type', 'Courier_Experience_yrs']
).ngroup() + 1

print("Delivery cleaned:", delivery.shape)
print(f"Delay flag distribution:\n{delivery['delay_flag'].value_counts()}")
delivery.head()

Delivery cleaned: (1000, 11)
Delay flag distribution:
delay_flag
1    664
0    336
Name: count, dtype: int64


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min,delay_flag,rider_id
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43,0,22
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84,1,3
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59,1,22
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37,0,22
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68,1,6


In [31]:
print("Delivery time percentiles:")
print(delivery['Delivery_Time_min'].quantile([0.25, 0.50, 0.75, 0.90]))

print("\nCounts by threshold:")
for t in [30, 45, 60]:
    late = (delivery['Delivery_Time_min'] > t).sum()
    print(f"  > {t} mins: {late} orders ({late/10:.1f}%)")

Delivery time percentiles:
0.25    41.0
0.50    55.5
0.75    71.0
0.90    86.0
Name: Delivery_Time_min, dtype: float64

Counts by threshold:
  > 30 mins: 882 orders (88.2%)
  > 45 mins: 664 orders (66.4%)
  > 60 mins: 415 orders (41.5%)


clean Customers dataset

In [24]:
# encode Output and Feedback
customers['Output'] = customers['Output'].map({'Yes': 1, 'No': 0})
customers['Feedback'] = customers['Feedback'].map({'Positive': 1, 'Negative': 0})

# add surrogate customer_id
customers.insert(0, 'customer_id', range(1, len(customers) + 1))

# rename to warehouse schema
customers.rename(columns={
    'Marital Status'              : 'marital_status',
    'Monthly Income'              : 'monthly_income',
    'Educational Qualifications'  : 'education',
    'Family size'                 : 'family_size',
    'Pin code'                    : 'pin_code',
    'Output'                      : 'orders_online',
    'Feedback'                    : 'feedback'
}, inplace=True)

print("Customers cleaned:", customers.shape)
customers.head(3)

Customers cleaned: (388, 13)


,customer_id,Age,Gender,marital_status,Occupation,monthly_income,education,family_size,latitude,longitude,pin_code,orders_online,feedback
0,1,20,Female,Single,Student,No Income,Post Graduate,4,12.9766,77.5993,560001,1,1.0
1,2,24,Female,Single,Student,Below Rs.10000,Graduate,3,12.9770,77.5773,560009,1,1.0
2,3,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,12.9551,77.6593,560017,1,NaN


build Dim_Rider

In [25]:
dim_rider = delivery.groupby(
    ['rider_id', 'Vehicle_Type', 'Courier_Experience_yrs']
).agg(
    avg_delivery_time=('Delivery_Time_min', 'mean'),
    total_deliveries=('Order_ID', 'count')
).reset_index()

dim_rider.rename(columns={
    'Vehicle_Type'           : 'vehicle_type',
    'Courier_Experience_yrs' : 'experience_yrs'
}, inplace=True)

print("Dim_Rider shape:", dim_rider.shape)
dim_rider.head(5)

Dim_Rider shape: (30, 5)


,rider_id,vehicle_type,experience_yrs,avg_delivery_time,total_deliveries
0,1,Bike,0.0,60.469388,49
1,2,Bike,1.0,60.578947,57
2,3,Bike,2.0,54.608696,46
3,4,Bike,3.0,59.812500,32
4,5,Bike,4.0,57.720930,43


  build Dim_Time

In [26]:
time_map = {'Morning': 9, 'Afternoon': 13, 'Evening': 18, 'Night': 22}

dim_time = delivery[['Time_of_Day']].drop_duplicates().reset_index(drop=True)
dim_time.insert(0, 'time_id', range(1, len(dim_time) + 1))
dim_time['hour'] = dim_time['Time_of_Day'].map(time_map)
dim_time['is_weekend'] = 0  # not available in source, default 0

print("Dim_Time:")
dim_time.head()

Dim_Time:


,time_id,Time_of_Day,hour,is_weekend
0,1,Afternoon,13,0
1,2,Evening,18,0
2,3,Night,22,0
3,4,Morning,9,0


build Dim_Location and export all CSVs

In [27]:
dim_location = customers[['customer_id', 'latitude', 'longitude', 'pin_code']].copy()
dim_location.rename(columns={'customer_id': 'location_id'}, inplace=True)

# add area and city from swiggy
dim_location['area'] = None
dim_location['city'] = 'Bangalore'

In [28]:
# export all clean CSVs
dim_restaurant.to_csv('dim_restaurant.csv', index=False)
customers.to_csv('dim_customer.csv', index=False)
dim_rider.to_csv('dim_rider.csv', index=False)
dim_time.to_csv('dim_time.csv', index=False)
dim_location.to_csv('dim_location.csv', index=False)
delivery.to_csv('fact_orders_ready.csv', index=False)

print("All CSVs exported successfully!")
print("\nFile summary:")
print(f"  dim_restaurant.csv  : {len(dim_restaurant)} rows")
print(f"  dim_customer.csv    : {len(customers)} rows")
print(f"  dim_rider.csv       : {len(dim_rider)} rows")
print(f"  dim_time.csv        : {len(dim_time)} rows")
print(f"  dim_location.csv    : {len(dim_location)} rows")
print(f"  fact_orders_ready.csv: {len(delivery)} rows")

All CSVs exported successfully!

File summary:
  dim_restaurant.csv  : 15785 rows
  dim_customer.csv    : 388 rows
  dim_rider.csv       : 30 rows
  dim_time.csv        : 4 rows
  dim_location.csv    : 388 rows
  fact_orders_ready.csv: 1000 rows
